# Lab calc_01 — Numerical derivatives & gradient checking

**Track 1 · Calculus** — skills: *Gradients & directional derivatives*, *Multivariable chain rule & Jacobians*.

Every time you derive a gradient by hand (or write a backward pass), you can be wrong in a way that still
"trains a bit". The cure is the **gradient check**: compare your analytic gradient against a numerical
estimate built from nothing but function evaluations.

### Learning goals
1. Implement the **central difference** $f'(x) \approx \dfrac{f(x+h) - f(x-h)}{2h}$ and see why its error is $O(h^2)$
   while the forward difference $\frac{f(x+h)-f(x)}{h}$ is only $O(h)$.
2. Build a **numerical gradient** for $f:\mathbb R^{n}\to\mathbb R$ (any array shape), one coordinate at a time.
3. Use the **relative error** $\dfrac{|a-b|}{|a|+|b|}$ to decide whether two gradients agree.
4. Verify hand-derived gradients: a non-symmetric quadratic, logistic regression, and backprop through a tiny MLP.

### Reading
- CS231n notes, *Gradient checks*: <https://cs231n.github.io/neural-networks-3/>
- MIT 18.S096 *Matrix Calculus for Machine Learning and Beyond*: <https://ocw.mit.edu/courses/18-s096-matrix-calculus-for-machine-learning-and-beyond-january-iap-2023/>

Fill in every function marked `# TODO`, run the exploration cells, then run the final grading cell.

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

## 1. Central differences

Taylor-expand both sides:
$$f(x\pm h) = f(x) \pm h f'(x) + \tfrac{h^2}{2} f''(x) \pm \tfrac{h^3}{6} f'''(x) + \dots$$
Subtracting, the even terms cancel:
$$\frac{f(x+h)-f(x-h)}{2h} = f'(x) + \tfrac{h^2}{6} f'''(x) + O(h^4).$$

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    """Central difference estimate of f'(x) for a scalar function of a scalar."""
    # TODO
    raise NotImplementedError

**Explore.** Plot (or print) the error of forward vs. central differences for $f = \sin$ at $x=1$ as $h$ shrinks
from $10^{-1}$ to $10^{-12}$. You should see the $O(h)$ / $O(h^2)$ slopes — and then round-off error taking over
for tiny $h$ (the error grows like $\varepsilon_{\text{machine}}/h$). Where is the sweet spot for each?

In [ ]:
for h in [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8, 1e-10, 1e-12]:
    fwd = (np.sin(1 + h) - np.sin(1)) / h
    cen = numerical_derivative(np.sin, 1.0, h)
    print(f"h={h:.0e}  forward err={abs(fwd - np.cos(1)):.2e}  central err={abs(cen - np.cos(1)):.2e}")

## 2. Numerical gradient of $f:\mathbb R^{n}\to\mathbb R$

Perturb one coordinate at a time: $\;\dfrac{\partial f}{\partial x_i} \approx \dfrac{f(x + h e_i) - f(x - h e_i)}{2h}$.
Your function must work for **any array shape** (vectors, matrices of weights, ...) and must **not modify** the
caller's array (copy it, or restore every entry you touch). Hint: `np.nditer(x, flags=["multi_index"])` or loop over `x.size` with `x.flat`.

In [ ]:
def numerical_gradient(f, x, h=1e-5):
    """Central-difference gradient of scalar f at array x; returns an array of x's shape."""
    # TODO
    raise NotImplementedError


def relative_error(a, b, eps=1e-12):
    """Elementwise |a - b| / max(|a| + |b|, eps), maximised over all entries (return a float)."""
    # TODO
    raise NotImplementedError


def grad_check(f, grad_f, x, h=1e-5, tol=1e-6):
    """Return (ok, rel_err): compare grad_f(x) with numerical_gradient(f, x)."""
    # TODO
    raise NotImplementedError

In [ ]:
# sanity check on f(x, y) = x^2 y + sin(y): grad = (2xy, x^2 + cos y)
f = lambda v: v[0] ** 2 * v[1] + np.sin(v[1])
g = lambda v: np.array([2 * v[0] * v[1], v[0] ** 2 + np.cos(v[1])])
print(grad_check(f, g, np.array([1.5, -0.4])))

## 3. Gradients you derived on paper

**(a) Quadratic.** For $f(\mathbf x) = \tfrac12\mathbf x^\top A\mathbf x - \mathbf b^\top\mathbf x$ with a *non-symmetric* $A$,
show on paper that $\nabla f = \tfrac12(A + A^\top)\mathbf x - \mathbf b$. (If $A$ is symmetric this is the familiar $A\mathbf x - \mathbf b$.)

**(b) Logistic regression.** With $\sigma(s) = 1/(1+e^{-s})$, $\mathbf p = \sigma(X\mathbf w)$ and mean cross-entropy
$$\mathcal L(\mathbf w) = \frac1n\sum_k -y_k\ln p_k - (1-y_k)\ln(1-p_k),$$
show that $\nabla\mathcal L = \frac1n X^\top(\mathbf p - \mathbf y)$ (use $\sigma' = \sigma(1-\sigma)$).

In [ ]:
def grad_quadratic(A, b, x):
    """Gradient of 0.5 x^T A x - b^T x (A square, not necessarily symmetric)."""
    # TODO
    raise NotImplementedError


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logistic_loss(w, X, y):
    p = sigmoid(X @ w)
    return float(np.mean(-y * np.log(p) - (1 - y) * np.log(1 - p)))


def grad_logistic_loss(w, X, y):
    """Analytic gradient of logistic_loss with respect to w."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
A, b, x = rng.normal(size=(3, 3)), rng.normal(size=3), rng.normal(size=3)
print("quadratic:", grad_check(lambda v: 0.5 * v @ A @ v - b @ v, lambda v: grad_quadratic(A, b, v), x))
print("buggy 'A x - b' would give:", grad_check(lambda v: 0.5 * v @ A @ v - b @ v, lambda v: A @ v - b, x))
X, y = rng.normal(size=(10, 3)), (rng.random(10) > 0.5).astype(float)
print("logistic:", grad_check(lambda w: logistic_loss(w, X, y), lambda w: grad_logistic_loss(w, X, y), rng.normal(size=3)))

## 4. Backprop through a tiny MLP

$$\mathbf z = W_1\mathbf x + \mathbf b_1,\quad \mathbf h = \tanh(\mathbf z),\quad \hat y = \mathbf w_2^\top\mathbf h + b_2,\quad
\mathcal L = \tfrac12(\hat y - y)^2.$$

Derive on paper (multivariable chain rule, $\tanh' = 1 - \tanh^2$):
$$\delta = \hat y - y,\quad \frac{\partial\mathcal L}{\partial\mathbf w_2} = \delta\,\mathbf h,\quad
\frac{\partial\mathcal L}{\partial b_2} = \delta,\quad
\frac{\partial\mathcal L}{\partial\mathbf z} = \delta\,\mathbf w_2\odot(1-\mathbf h^2),\quad
\frac{\partial\mathcal L}{\partial W_1} = \frac{\partial\mathcal L}{\partial\mathbf z}\,\mathbf x^\top,\quad
\frac{\partial\mathcal L}{\partial\mathbf b_1} = \frac{\partial\mathcal L}{\partial\mathbf z}.$$

Shapes: `W1` is (H, D), `b1` (H,), `W2` (H,), `b2` a scalar (0-d array). Return gradients with the **same shapes**.

In [ ]:
def mlp_loss(params, x, y):
    h = np.tanh(params["W1"] @ x + params["b1"])
    yhat = params["W2"] @ h + params["b2"]
    return float(0.5 * (yhat - y) ** 2)


def mlp_grads(params, x, y):
    """Return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2} for mlp_loss."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(1)
params = {"W1": rng.normal(size=(5, 3)), "b1": rng.normal(size=5), "W2": rng.normal(size=5), "b2": np.array(0.1)}
x, y = rng.normal(size=3), -0.4
grads = mlp_grads(params, x, y)
for key in params:
    def f(v, key=key):
        p = dict(params); p[key] = v
        return mlp_loss(p, x, y)
    print(key, grad_check(f, lambda v, key=key: grads[key], params[key]))

### Reflection (write on paper)
- Why does the check use a *relative* error rather than $\|a-b\|$?
- Your gradient check passes at a random point but the network still diverges. What else could be wrong
  (kinks like ReLU at 0, `float32`, a step size issue)?
- Numerical gradients cost $2n$ function evaluations. Why do we still train with backprop?

## Grade

In [ ]:
from lc_client import grade
grade("calc_01_gradcheck", globals())